<a href="https://colab.research.google.com/github/SaimNaveed646/CardiovasularDiseasePrediction/blob/main/CardioDiseasePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [392]:
import pandas as pd

In [393]:
data = pd.read_csv("https://raw.githubusercontent.com/SaimNaveed646/CardiovasularDiseasePrediction/main/cardio_train.csv", sep=";", header=0)

In [394]:
data.head()


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [395]:
data.shape

(70000, 13)

In [396]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [397]:
data.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,49972.419900,19468.865814,1.349571,164.359229,74.205690,128.817286,96.630414,1.366871,1.226457,0.088129,0.053771,0.803729,0.499700
std,28851.302323,2467.251667,0.476838,8.210126,14.395757,154.011419,188.472530,0.680250,0.572270,0.283484,0.225568,0.397179,0.500003
min,0.000000,10798.000000,1.000000,55.000000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,25006.750000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,50001.500000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,74889.250000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,99999.000000,23713.000000,2.000000,250.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000


In [398]:
data['cardio'].nunique()          # total classes


2

In [399]:
data['cardio'].value_counts()

,count
cardio,
0,35021
1,34979


In [400]:
data['cardio'].value_counts(normalize=True) * 100

,proportion
cardio,
0,50.03
1,49.97


# **Cleaning**

In [401]:
data_feat = data.copy()

In [402]:
data_feat['id'].duplicated().sum()

np.int64(0)

In [403]:
data_feat.isnull().sum()

,0
id,0
age,0
gender,0
height,0
weight,0
ap_hi,0
ap_lo,0
cholesterol,0
gluc,0
smoke,0


In [404]:
data_feat['age'] = data_feat['age'] / 365

In [405]:
data_feat['height'] = data_feat['height'] / 100

In [406]:
data_feat = data_feat[
    (data['ap_hi'] >= 50) &
    (data['ap_hi'] <= 250) &
    (data['ap_lo'] >= 40) &
    (data['ap_lo'] <= 200) &
    (data['ap_hi'] > data['ap_lo'])
]

In [407]:
data_feat.shape

(68672, 13)

# **Feature Engineering**

In [408]:
data_feat["bmi"] = data_feat["weight"] / (data_feat["height"] ** 2)

In [409]:
def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

data_feat['BMI_Category'] = data_feat['bmi'].apply(bmi_category)

In [410]:
data_feat['BMI_Category'].value_counts()

,count
BMI_Category,
Normal,25425
Overweight,24624
Obese,17980
Underweight,643


In [411]:
pd.crosstab(data_feat['BMI_Category'], data_feat['cardio'])

cardio,0,1
BMI_Category,,
Normal,15309,10116
Obese,6748,11232
Overweight,12176,12448
Underweight,466,177


In [412]:
def age_group(age):
  if age < 25:
    return "Young"
  elif age < 45 :
    return "adult"
  elif age < 60:
    return "Middle-aged"
  else:
    return "Senior"

In [413]:
data_feat['age_group'] = data_feat['age'].apply(age_group)

In [414]:
pd.crosstab(data_feat['age_group'], data_feat['cardio'])

cardio,0,1
age_group,,
Middle-aged,23299,22492
Senior,4255,8524
adult,7145,2957


In [415]:
data_feat.columns

Index(['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo',
       'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio', 'bmi',
       'BMI_Category', 'age_group'],
      dtype='object')

In [416]:
data_feat.head()


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,bmi,BMI_Category,age_group
0,0,50.391781,2,1.68,62.0,110,80,1,1,0,0,1,0,21.967120,Normal,Middle-aged
1,1,55.419178,1,1.56,85.0,140,90,3,1,0,0,1,1,34.927679,Obese,Middle-aged
2,2,51.663014,1,1.65,64.0,130,70,3,1,0,0,0,1,23.507805,Normal,Middle-aged
3,3,48.282192,2,1.69,82.0,150,100,1,1,0,0,1,1,28.710479,Overweight,Middle-aged
4,4,47.873973,1,1.56,56.0,100,60,1,1,0,0,0,0,23.011177,Normal,Middle-aged


# **Feature Scaling**

In [417]:
data_feat.drop(columns=['id', 'age', 'height', 'weight','bmi'], inplace=True)

In [418]:
data_feat.head()

,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,BMI_Category,age_group
0,2,110,80,1,1,0,0,1,0,Normal,Middle-aged
1,1,140,90,3,1,0,0,1,1,Obese,Middle-aged
2,1,130,70,3,1,0,0,0,1,Normal,Middle-aged
3,2,150,100,1,1,0,0,1,1,Overweight,Middle-aged
4,1,100,60,1,1,0,0,0,0,Normal,Middle-aged


In [419]:
X=data_feat[["gender","ap_hi","ap_lo","cholesterol","gluc","smoke","alco","active","BMI_Category","age_group"]]
y=data_feat["cardio"]

In [420]:
X.head(5)

,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,BMI_Category,age_group
0,2,110,80,1,1,0,0,1,Normal,Middle-aged
1,1,140,90,3,1,0,0,1,Obese,Middle-aged
2,1,130,70,3,1,0,0,0,Normal,Middle-aged
3,2,150,100,1,1,0,0,1,Overweight,Middle-aged
4,1,100,60,1,1,0,0,0,Normal,Middle-aged


In [421]:
y.head(5)

,cardio
0,0
1,1
2,1
3,1
4,0


In [422]:
categorical_features = ['gender', 'BMI_Category', 'age_group',"cholesterol","gluc","smoke","alco","active"]
numeric_features =  ["ap_hi","ap_lo"]

In [423]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Create ColumnTransformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

In [424]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [425]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [426]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['ap_hi', 'ap_lo']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['gender', 'BMI_Category',
                                                   'age_group', 'cholesterol',
                                                   'gluc', 'smoke', 'alco',
                                                   'active'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [427]:
from sklearn.metrics import accuracy_score
y_pred_train = pipeline.predict(X_test)
accuracy_score(y_test, y_pred_train)

0.7118310884601383

In [428]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_train))

              precision    recall  f1-score   support

           0       0.70      0.76      0.73      6940
           1       0.73      0.67      0.70      6795

    accuracy                           0.71     13735
   macro avg       0.71      0.71      0.71     13735
weighted avg       0.71      0.71      0.71     13735



In [429]:
import pandas as pd

X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

In [430]:
X_train.head()

,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,BMI_Category,age_group
13011,1,120,79,1,1,0,0,1,Normal,Middle-aged
24119,1,140,90,2,1,0,0,1,Normal,adult
59987,2,130,90,1,1,0,0,1,Overweight,Middle-aged
42827,1,120,79,2,1,0,0,0,Normal,Middle-aged
51528,1,120,90,1,1,0,0,1,Obese,Middle-aged


In [431]:
X_train.columns

Index(['gender', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco',
       'active', 'BMI_Category', 'age_group'],
      dtype='object')

In [432]:
import pickle

# File path variable
pickle_model_path = "model.pkl"

# Save the model/pipeline
with open(pickle_model_path, "wb") as file:
    pickle.dump(pipeline, file)

**Logistic Regression**

In [433]:
# from sklearn.linear_model import LogisticRegression

# log_model = LogisticRegression(random_state=42, max_iter=1000)

# log_model.fit(X_train, y_train)

# y_pred_log = log_model.predict(X_test)

In [434]:
# from sklearn.metrics import classification_report

# print(classification_report(y_test, y_pred_log))